<a href="https://colab.research.google.com/github/WTavasi/computing-for-bioinformatics/blob/main/166082_traversal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Path Traversal in Biological Networks using BFS and DFS

**Name:** Tavasi Wyclef Wasike
**Admission number:** 166082

## Setup: the gene network

In [1]:
# Gene interaction network (undirected)
graph = {
    0: [1, 3],
    1: [0, 2],
    2: [1],
    3: [0, 4, 7],
    4: [3, 5, 6, 7],
    5: [4, 6],
    6: [4, 5, 7],
    7: [3, 4, 6],
}
SOURCE = 0
DESTINATION = 7

# These dictionaries remember which gene each gene was reached from.
bfs_parents = {}
dfs_parents = {}

## Part A: BFS traversal

BFS uses a **queue** (first in, first out).

In [2]:
def bfs(graph, start):
    visited = []
    queue = [start]
    seen = [start]
    bfs_parents[start] = None

    iteration = 1
    while len(queue) > 0:
        current = queue.pop(0)
        visited.append(current)

        print("BFS iteration", iteration)
        print("  Current gene:", current)

        # look at the neighbours, smallest first (the lists are already sorted)
        for neighbour in graph[current]:
            if neighbour not in seen:
                seen.append(neighbour)
                queue.append(neighbour)
                bfs_parents[neighbour] = current

        print("  Visited so far:", visited)
        print("  Queue now:", queue)
        iteration = iteration + 1

    return visited

## Part B: DFS traversal

In [3]:
def dfs_visit(graph, current, visited):
    # visit this gene
    visited.append(current)
    print("DFS iteration", len(visited))
    print("  Current gene:", current)
    print("  Visited so far:", visited)

    # go deep into the first unvisited neighbour, then the next, and so on
    for neighbour in graph[current]:
        if neighbour not in visited:
            dfs_parents[neighbour] = current
            dfs_visit(graph, neighbour, visited)


def dfs(graph, start):
    visited = []
    dfs_parents[start] = None
    dfs_visit(graph, start, visited)
    return visited

## Helper for Part D: rebuild a path

In [4]:
def build_path(parents, start, end):
    path = []
    gene = end
    while gene is not None:
        path.append(gene)
        gene = parents[gene]
    path.reverse()
    return path

## Part C: Run both algorithms from gene 0

In [5]:
print("========== BFS ==========")
bfs_path = bfs(graph, SOURCE)
print()
print("========== DFS ==========")
dfs_path = dfs(graph, SOURCE)
print()

print("Final BFS traversal path:", bfs_path)
print("Final DFS traversal path:", dfs_path)

========== BFS ==========
BFS iteration 1
  Current gene: 0
  Visited so far: [0]
  Queue now: [1, 3]
BFS iteration 2
  Current gene: 1
  Visited so far: [0, 1]
  Queue now: [3, 2]
BFS iteration 3
  Current gene: 3
  Visited so far: [0, 1, 3]
  Queue now: [2, 4, 7]
BFS iteration 4
  Current gene: 2
  Visited so far: [0, 1, 3, 2]
  Queue now: [4, 7]
BFS iteration 5
  Current gene: 4
  Visited so far: [0, 1, 3, 2, 4]
  Queue now: [7, 5, 6]
BFS iteration 6
  Current gene: 7
  Visited so far: [0, 1, 3, 2, 4, 7]
  Queue now: [5, 6]
BFS iteration 7
  Current gene: 5
  Visited so far: [0, 1, 3, 2, 4, 7, 5]
  Queue now: [6]
BFS iteration 8
  Current gene: 6
  Visited so far: [0, 1, 3, 2, 4, 7, 5, 6]
  Queue now: []

========== DFS ==========
DFS iteration 1
  Current gene: 0
  Visited so far: [0]
DFS iteration 2
  Current gene: 1
  Visited so far: [0, 1]
DFS iteration 3
  Current gene: 2
  Visited so far: [0, 1, 2]
DFS iteration 4
  Current gene: 3
  Visited so far: [0, 1, 2, 3]
DFS iteration 

## Part D: Path from gene 0 to gene 7

In [6]:
bfs_route = build_path(bfs_parents, SOURCE, DESTINATION)
dfs_route = build_path(dfs_parents, SOURCE, DESTINATION)

# a path with N genes has N - 1 edges
print("BFS path from 0 to 7:", bfs_route, "->", len(bfs_route) - 1, "interactions")
print("DFS path from 0 to 7:", dfs_route, "->", len(dfs_route) - 1, "interactions")

BFS path from 0 to 7: [0, 3, 7] -> 2 interactions
DFS path from 0 to 7: [0, 3, 4, 5, 6, 7] -> 5 interactions


## Part E: Compare BFS and DFS

BFS visited the genes in the order 0, 1, 3, 2, 4, 7, 5, 6. DFS visited them in the order 0, 1, 2, 3, 4, 5, 6, 7.

**BFS and the queue.** BFS uses a queue (first in, first out). The neighbours of gene 0 (genes 1 and 3) join the queue first, so both are visited before anything further away. The search moves outward one interaction at a time, in layers: distance 1 (genes 1, 3), then distance 2 (genes 2, 4, 7), then distance 3 (genes 5, 6).

**DFS and the stack.** DFS uses a stack (here, the recursion). It always continues from the newest gene it found. From gene 0 it goes to 1, then 2. Gene 2 is a dead end, so it backtracks to gene 0 and goes to 3, then 4, 5 and 6, and reaches 7 last. It goes deep along one route before trying another.

**Which path was shorter?** BFS gave the shorter path from gene 0 to gene 7. Its path 0 -> 3 -> 7 has 2 interactions, while the DFS path 0 -> 3 -> 4 -> 5 -> 6 -> 7 has 5 interactions. BFS always finds the path with the fewest interactions because it checks every gene at distance 1 before distance 2, and so on. DFS takes whichever route it happens to go down first, which is not always the shortest.

## Part F: Reflection

**BFS** can represent a signal spreading outward one step at a time, so it suits questions about the shortest communication route between two genes. Example question for BFS: "What is the shortest chain of protein interactions that carries a signal from a receptor protein to a target gene?" Fewer steps often means a faster signal with fewer chances for error.

**DFS** can represent following one full pathway as far as it goes before trying another branch, so it suits questions about exploring a whole pathway and finding which genes can be reached at all. Example question for DFS: "Which genes and proteins are downstream of this gene in a regulatory cascade, and what does one complete pathway from start to finish look like?"